# Colab A100 — SFT Readiness (Cursor Colab extension)

Full runbook: `docs/runbook_cursor_colab_ext_sft_readiness.md`

**Connect first:** kernel selector (top-right) → **Colab** → sign in as
`eric.wu@alphaaiengineering.com` → **Auto Connect** to the A100 (or *New Colab Server* → A100).
Then run cells top-to-bottom.

**Flow:** Steps 1–3 → ⏸ **PAUSE for approval before Step 4** (~9.85 GB pull) → Steps 5–6 → ⛔ **STOP at Step 7** (plan mode).

**Constraints:** frozen tokenizer is immutable; corpus (`data/`, `luts/`) is read-only; ask before the pull / any training / any HF push.

> Note: the extension runs these cells on the **remote A100**, but the filesystem is the remote VM (`/content`) — your local repo `.env` is *not* visible here (Step 3 handles the token).

## Step 1 — Verify runtime (no approval needed)
Confirm: an **A100** (40/80 GB), Python 3.1x, tens of GB free on `/content`.

In [ ]:
!nvidia-smi
import sys; print("PY", sys.version)
!df -h /content

## Step 2 — Clone + install  ⚠ longish install
Use `[ml]` (torch/CUDA), **not** `[mlx]` (Apple-only).

In [ ]:
!git clone https://github.com/ericrcwu001/SLM
!cd SLM && pip install -e '.[ml]'

## Step 3 — HF auth (extension-ready; **replaces** `userdata.get`)
The extension has **no Secrets manager** and can't see your local repo `.env`. This loader resolves
`HF_TOKEN` from, in order: an existing env var → a `.env` **uploaded** to `/content/SLM/.env` or
`/content/.env` → a masked `getpass` paste. It never prints the token or bakes it into the notebook.

To skip the paste, upload the repo `.env` into `/content/SLM/` via Cursor's remote file explorer first.
⚠️ That file also holds Kaggle/freshluts secrets — upload an `HF_TOKEN`-only copy if you'd rather not
put those on the VM. A **read-only** token is enough for staging.

In [ ]:
import os, getpass
from pathlib import Path

def _parse_env_file(path):
    vals = {}
    for raw in Path(path).read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export "):]
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        vals[k.strip()] = v.strip().strip('"').strip("'")
    return vals

def ensure_hf_token():
    if os.environ.get("HF_TOKEN"):
        return "already in os.environ"
    for p in ("/content/SLM/.env", "/content/.env", ".env"):
        if Path(p).exists():
            v = _parse_env_file(p).get("HF_TOKEN")
            if v:
                os.environ["HF_TOKEN"] = v
                return "loaded from " + p
    tok = getpass.getpass("Paste HF_TOKEN (input hidden; copy from repo .env): ").strip()
    if tok:
        os.environ["HF_TOKEN"] = tok
        return "loaded from getpass prompt"
    return None

src = ensure_hf_token()
print("HF_TOKEN source:", src)
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set - upload repo .env to the runtime or paste it." 

## Step 4 — Stage the corpus  ⏸ DO NOT RUN until approved (~9.85 GB pull)
Keep Cursor connected for the whole pull. Confirm: **all 5 shards sha256-verified** and `/content/slm`
contains `luts/`, `data/`, `tokenizer/final/`.

In [ ]:
# STOP: ~9.85 GB PULL - run only after explicit approval.
import os
!slm_stage stage --durable-root hf://datasets/ericrcwu/LUT_SLM --local-root /content/slm
os.environ["SLM_ARTIFACT_ROOT"] = "/content/slm"
print("SLM_ARTIFACT_ROOT =", os.environ["SLM_ARTIFACT_ROOT"])
!ls -la /content/slm

## Step 5 — VERIFY THE FROZEN TOKENIZER (hard gate)
**If either assert fails → STOP and report. Do not proceed.**

In [ ]:
import json, pathlib
d = pathlib.Path("/content/slm/tokenizer/final")
need = {"model.pt", "encoder.pt", "decoder.pt", "codebook.npy", "manifest.json"}
have = {p.name for p in d.iterdir()}
assert need <= have, f"MISSING weights: {sorted(need - have)}"

m = json.load(open(d / "manifest.json"))
EXPECT_VER = "vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f"
EXPECT_SHA = "bcdf369dd7cd9a99d71f240b0dac67d404f52130dc8c35d14d6a04514349d118"
assert m.get("tokenizer_version")  == EXPECT_VER, ("VERSION MISMATCH", m.get("tokenizer_version"))
assert m.get("vq_codebook_sha256") == EXPECT_SHA, ("SHA MISMATCH",     m.get("vq_codebook_sha256"))
print("FROZEN TOKENIZER OK:", m["tokenizer_version"])

### Step 5b (optional) — copy staged tokenizer into the repo
Only if you prefer copying over resolving via `SLM_ARTIFACT_ROOT`. Ensures SFT/tokenize code loads the
**staged frozen** weights, never the weightless cloned stub.

In [ ]:
import shutil, pathlib
src = pathlib.Path("/content/slm/tokenizer/final")
dst = pathlib.Path("/content/SLM/tokenizer/final"); dst.mkdir(parents=True, exist_ok=True)
for p in src.iterdir(): shutil.copy2(p, dst / p.name)
print("copied:", sorted(x.name for x in dst.iterdir()))

## Step 6 — Read docs, scope tokenizer→SFT prerequisites (read-only)

In [ ]:
for f in ["docs/training_plan_colab.md", "docs/master_plan.md", "docs/model_architecture.md"]:
    print("\n" + "="*80 + f"\n{f}\n" + "="*80)
    print(open(f"/content/SLM/{f}").read())

## Step 7 — STOP for plan approval  ⛔
No training, no wiring stubs, no edits to `data/`/`luts/`. I'll produce the **readiness report** +
**first-SFT-run plan** (levers: cap `max_pixels`, raise batch size, keep `epochs=2`) in **plan mode**
for your approval before implementing anything.